In [1]:
from datetime import datetime
import logging
import shutil
import sys

from deep_notes_ai.config.logging_setup import configure_logging
from deep_notes_ai.config.settings import Settings
from deep_notes_ai.langgraph_pipeline.graph import build_graph
from deep_notes_ai.domain.models import SourceType


def run_pipeline(
    youtube_url: str,
    source_type: SourceType
):
    
    try:
        settings = Settings()
    except Exception as exc:
        print(f"[ERROR] Failed to load settings: {exc}", file=sys.stderr)
        return 1

    logger_name = "deep_notes_ai"
    log_file_path = configure_logging(
        logger_name=logger_name,
        log_level=settings.log_level,
        structured=settings.enable_structured_logging,
        log_dir=settings.logs_dir,
    )

    try:
        graph, monitor_service = build_graph(settings)
    except Exception as exc:
        print(f"[ERROR] Failed to build pipeline graph: {exc}", file=sys.stderr)
        return 1

    initial_state = {
        "source": youtube_url,
        "source_type": source_type,
        "pipeline_complete": False,
        "error_message": None,
    }

    print(f"[INFO] Starting pipeline for source: {youtube_url} (type: {source_type})")
    try:
        final_state = graph.invoke(
            initial_state,
            config={"configurable": {"thread_id": youtube_url}},
        )
        print("[INFO] Pipeline completed successfully.")
    except Exception as exc:
        print(f"[ERROR] Pipeline failed: {exc}", file=sys.stderr)
    
    if not final_state.get("pipeline_complete"):
        error_msg = final_state.get("error_message", "Unknown error")
        print(f"[ERROR] Pipeline did not complete successfully: {error_msg}", file=sys.stderr)

    print("\n[INFO] Pipeline final state:")
    print(f"      Source            : {final_state.get('source')}")
    print(f"      Source Type       : {final_state.get('source_type')}")
    print(f"      Run Directory     : {final_state.get('current_run_dir')}")
    print(f"      Content Id        : {final_state.get('content_id')}")
    print(f"      Content Title     : {final_state.get('content_title')}")
    print(f"      Author Name       : {final_state.get('author_name')}")
    print(f"      Upload Date       : {final_state.get('upload_date')}")
    print(f"      Content URL       : {final_state.get('content_url')}")
    print(f"      Nodes Count       : {final_state.get('content_node_count')}")
    print(f"      Pipeline Complete : {final_state.get('pipeline_complete')}")
    print(f"      Error Message     : {final_state.get('error_message')}")

    run_dir = final_state.get("current_run_dir")
    if run_dir is not None:
        if monitor_service is not None:
            print("\n[INFO] Saving LLM monitoring report.")
            try:
                monitor_service.save_reports(run_dir)
                print("[INFO] LLM monitoring report saved successfully.")
            except Exception as exc:
                print(f"[ERROR] LLM monitoring report failed: {exc}", file=sys.stderr)
        
        print("\n[INFO] Moving pipeline logs.")
        try:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            destination_log = run_dir / "artifacts" / f"pipeline_{timestamp}.log"
            logger = logging.getLogger(logger_name)

            for handler in logger.handlers:
                handler.flush()
                handler.close()

            if log_file_path.exists():
                destination_log.parent.mkdir(parents=True, exist_ok=True)
                shutil.move(log_file_path, destination_log)
                print(f"[INFO] Pipeline log moved to: {destination_log}")
            else:
                print(f"[ERROR] Pipeline log not found: {log_file_path}", file=sys.stderr)
        except Exception as exc:
            print(f"[ERROR] Moving pipeline log failed: {exc}", file=sys.stderr)

    return 0

In [2]:
run_pipeline(
    youtube_url="https://www.youtube.com/watch?v=FepDo-0DrSo&list=PLZoTAELRMXVM8Pf4U67L4UuDRgV4TNX9D&index=7",
    source_type=SourceType.YOUTUBE
)


[INFO] Starting pipeline for source: https://www.youtube.com/watch?v=FepDo-0DrSo&list=PLZoTAELRMXVM8Pf4U67L4UuDRgV4TNX9D&index=7 (type: youtube)
────────────────────────────────────────────────
⟳ Extracting Video Metadata
✓ Extracting Video Metadata
⟳ Downloading Transcript
✓ Downloading Transcript
⟳ Cleaning Transcript
    Chunk 2 / 2  (2 / 2)                        
✓ Cleaning Transcript
⟳ Numbering Transcript
✓ Numbering Transcript
⟳ Generating Topic Hierarchy
✓ Generating Topic Hierarchy
⟳ Extracting Content Nodes
✓ Extracting Content Nodes
⟳ Generating Structured Content
    Partition 1 / 1  (1 / 1)                    
✓ Generating Structured Content
⟳ Generating Summaries
    Partition 2 / 2  (2 / 2)                    
✓ Generating Summaries
⟳ Rendering Markdown
✓ Rendering Markdown
· Pipeline finished — notes ready
[INFO] Pipeline completed successfully.

[INFO] Pipeline final state:
      Source            : https://www.youtube.com/watch?v=FepDo-0DrSo&list=PLZoTAELRMXVM8Pf4U67

0

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi


content_id = "jGg_1h0qzaM"

try:
    ytt_api = YouTubeTranscriptApi()
    fetched_transcript = ytt_api.fetch(content_id)

    transcript = " ".join(snippet.text for snippet in fetched_transcript)
    print(transcript)

except Exception as e:
    print(f"Error: {e}")


In [7]:
import json
import subprocess

content_id = "jGg_1h0qzaM"

result = subprocess.run(
    [
        "yt-dlp",
        "--skip-download",
        "--dump-single-json",
        f"https://www.youtube.com/watch?v={content_id}",
    ],
    capture_output=True,
    text=True,
)

data = json.loads(result.stdout)

description = data["description"]
title = data["title"]
duration = data["duration"]

In [ ]:
print(title)
print(description)
print(duration)

In [ ]:
from yt_dlp import YoutubeDL

content_id = "jGg_1h0qzaM"

url = f"https://www.youtube.com/watch?v={content_id}"

ydl_opts = {
    "quiet": True,
    "skip_download": True,
}

with YoutubeDL(ydl_opts) as ydl:
    info = ydl.extract_info(url, download=False)

description = info.get("description", "")
title = info.get("title", "")
duration = info.get("duration")

print(title)
print(duration)
print(description)

In [ ]:
from deep_notes_ai.config.settings import Settings
from deep_notes_ai.services.console_reporter import ConsoleReporter
from deep_notes_ai.services.llm_monitor_service import LLMMonitorService
from deep_notes_ai.services.llm_service import LLMService
from deep_notes_ai.services.persistence_service import PersistenceService
from deep_notes_ai.services.pricing_service import PricingService
from deep_notes_ai.services.progress_service import ProgressService
from langchain_core.runnables import Runnable

from deep_notes_ai.services.prompt_service import PromptService


settings = Settings()
configure_logging(
    log_level=settings.log_level,
    structured=settings.enable_structured_logging,
)
persistence_service = PersistenceService()
prompt_service = PromptService(settings.prompts_dir)
pricing_service = PricingService()
monitor_service: LLMMonitorService | None = None
monitor_service = LLMMonitorService(
    pricing_service=pricing_service,
    persistence_service=persistence_service,
)

console_reporter = ConsoleReporter()
progress_service = ProgressService(reporters=[console_reporter])

llm_service = LLMService(settings, monitor_service=monitor_service)

cleaning_prompt = prompt_service.load("yt_transcript_cleaner")
cleaning_model = llm_service.get_chat_model(
    provider=settings.cleaning_model_provider,
    model=settings.cleaning_model_name,
    temperature=settings.cleaning_model_temperature,
    node_name="NODE_CLEAN_TRANSCRIPT",
    operation_name="Transcript Cleaning",
)
cleaning_chain: Runnable = cleaning_prompt | cleaning_model